# CS781 Phase 7 — Prior Case Retrieval with our Phase-1 models

**Dual encoder:** every case (query or candidate) -> 8 windows of 512 tokens spread over the case -> Phase-1 backbone -> mean-pooled window vectors -> case vector (mean or attention over windows, projection, L2 norm); relevance = cosine similarity. **Training:** InfoNCE with in-batch negatives and BM25 hard negatives (lexically similar but not cited cases); other cases relevant to the same query are masked out. **Lexical baseline:** BM25 tuned on dev; **final ranking:** w x z(dense) + (1-w) x z(BM25) with w chosen on dev (w >= 0.2, so the Phase-1 model always contributes). Exactly 7 unique candidates from the split's own pool, never the query itself. Metric: micro-F1@7.

**Kaggle:** GPU T4, Internet on, secret `HF`/`HF_TOKEN`, the 4 Phase-1 model datasets.

In [ ]:
RUN_SET = "A"
SETS = {
    "A": [
        dict(ARCH="hybrid_prefixlm",  METHOD="full",  LR=2e-5),
        dict(ARCH="efficient_retnet", METHOD="kd", KD_TEACHER_RUN="hybrid_prefixlm_full"),
        dict(ARCH="hybrid_prefixlm",  METHOD="lorma", LR=2e-4),
        dict(ARCH="decoder_llama",    METHOD="ia3",   LR=3e-3),
    ],
    "B": [],
}
EXPERIMENTS = SETS[RUN_SET]
TIME_BUDGET_HOURS = 11.0
MAX_TRAIN_MINUTES = 100

MAX_CHUNKS = 8
TRAIN_CHUNKS = 4
CHUNK_LEN = 512
CHARS_PER_TOKEN = 5
POOL = "mean"
PROJ_DIM = 256
CONTEXT = None

TOP_K = 7
TAU = 0.05
HARD_NEG_TOP = 20
FUSION_W_GRID = [round(0.2 + 0.1 * i, 1) for i in range(9)]
BM25_GRID = dict(k1=[1.2, 2.0], b=[0.75, 0.9], ngram=[1, 2], qweight=["log", "binary"])

EPOCHS = 3
BATCH_QUERIES = 4
EMBED_BATCH_DOCS = 8
LR = None
HEAD_LR = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
GRAD_CLIP = 1.0
AMP = True
SEED = 42
LOG_EVERY = 50

PARTIAL_TOP_K = 4
RANK, ALPHA, PEFT_DROPOUT = 16, 32, 0.05
PEFT_TARGET_FFN = True
ADAPTER_DIM = 64
N_PROMPT = 0
KD_TEACHER_RUN = None
KD_TEACHER_DIR = None
KD_KEEP_LAYERS = [0, 2, 4, 6, 8, 10]
KD_ALPHA, KD_TEMPERATURE = 0.5, 2.0
SAVE_WEIGHTS = False

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
DATA_PREFIX = "Phase_7_PCR"
LOCAL_DATA_DIR = None
SMOKE_TEST = False

DEFAULT_CONTEXT = {"decoder_llama": "bidir", "efficient_retnet": "echo", "hybrid_prefixlm": "bidir", "encdec_t5": "bidir"}
DEFAULT_LR = {"full": 2e-5, "kd": 1e-4, "partial": 1e-4, "gradual": 5e-5, "lorma": 2e-4, "ia3": 3e-3, "prompt": 1e-2}
BASE_C = {k: v for k, v in dict(globals()).items() if k.isupper() and not k.startswith("_")}

def experiment_config(overrides):
    c = dict(BASE_C, **overrides)
    c["CONTEXT"] = overrides.get("CONTEXT") or DEFAULT_CONTEXT.get(c["ARCH"], "-")
    c["LR"] = overrides.get("LR") or DEFAULT_LR.get(c["METHOD"], 0.0)
    c["RUN_NAME"] = overrides.get("RUN_NAME") or f"{c['ARCH']}_{c['METHOD']}"
    if c["CONTEXT"] == "echo": assert c["CHUNK_LEN"] <= 640
    return c

C = experiment_config(EXPERIMENTS[0])
print(f"RUN SET {RUN_SET}")
for _i, _e in enumerate(EXPERIMENTS, 1):
    _c = experiment_config(_e)
    print(f"{_i}. {_c['RUN_NAME']:32s} arch={_c['ARCH']:16s} method={_c['METHOD']:7s} lr={_c['LR']}")

In [ ]:
import os, sys, json, glob, time, random, shutil, zipfile, hashlib, math, re, subprocess
from collections import Counter
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")
if device.type != "cuda" and not SMOKE_TEST:
    raise RuntimeError("NO GPU: this session runs on CPU (training would take > 12 h). In Kaggle: Settings -> Accelerator -> "
                       "GPU T4 x2 (check your weekly GPU quota), then run all cells again.")
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()

def hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    for name in ("HF_TOKEN", "HF"):
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret(name)
            if tok: return tok.strip()
        except Exception:
            pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

HF_TOKEN_VALUE = hf_token()
if HF_TOKEN_VALUE:
    os.environ["HF_TOKEN"] = HF_TOKEN_VALUE
    print("Hugging Face token found (length", len(HF_TOKEN_VALUE), ")")
else:
    print("WARNING: no Hugging Face token found. On Kaggle: Add-ons -> Secrets -> add 'HF_TOKEN' (or 'HF') "
          "and switch its toggle ON for this notebook; the course dataset is gated.")

def data_file(rel):
    roots = [LOCAL_DATA_DIR] if LOCAL_DATA_DIR else []
    for root in roots:
        p = os.path.join(root, rel)
        if os.path.exists(p): return p
    hits = glob.glob(f"/kaggle/input/**/{DATA_PREFIX}/{rel}", recursive=True)
    if hits: return hits[0]
    from huggingface_hub import hf_hub_download
    assert HF_TOKEN_VALUE, ("No Hugging Face token: add the Kaggle secret HF_TOKEN (or HF), switch it ON for this notebook "
                           "(Add-ons -> Secrets), then run again.")
    return hf_hub_download(HF_DATASET_REPO, f"{DATA_PREFIX}/{rel}", repo_type="dataset", token=HF_TOKEN_VALUE)

def find_model_dir(arch):
    for cfgp in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
        try:
            if json.load(open(cfgp)).get("arch") == arch and os.path.exists(os.path.join(os.path.dirname(cfgp), "pytorch_model.bin")):
                return os.path.dirname(cfgp)
        except Exception:
            pass
    raise FileNotFoundError(f"no Phase-1 model folder for arch={arch} under /kaggle/input "
                            "(attach the dataset that contains config.json + pytorch_model.bin + legal_bpe_tokenizer.json)")

### Phase-1 code (unchanged)

In [ ]:
import heapq
import json
import os
import re
import time
from collections import Counter, defaultdict

PRETOKENIZE_PATTERN = (
    r"'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?(?:[^\s\w]|_)+|\s+(?!\S)|\s+"
)
EOS_TOKEN = "<|endoftext|>"

class LegalBPETokenizer:
    def __init__(self, merges=None, name="legal-bpe"):
        self.name = name
        self.merges = [tuple(m) for m in (merges or [])]
        self._pattern = re.compile(PRETOKENIZE_PATTERN)
        self._build_tables()

    def _build_tables(self):
        self.ranks = {pair: i for i, pair in enumerate(self.merges)}
        self.vocab = {i: bytes([i]) for i in range(256)}
        for i, (a, b) in enumerate(self.merges):
            self.vocab[256 + i] = self.vocab[a] + self.vocab[b]
        self.eos_token = EOS_TOKEN
        self.eos_token_id = 256 + len(self.merges)
        self.pad_token = self.eos_token
        self.pad_token_id = self.eos_token_id
        self.model_max_length = 10**9
        self._cache = {}

    def __len__(self):
        return 256 + len(self.merges) + 1

    @property
    def vocab_size(self):
        return len(self)

    def pretokenize(self, text):
        return self._pattern.findall(text)

    @classmethod
    def train(cls, texts, vocab_size, name="legal-bpe", min_frequency=2, verbose=True):
        assert 258 <= vocab_size <= 65535, "vocab_size must fit in uint16"
        t0 = time.time()
        pattern = re.compile(PRETOKENIZE_PATTERN)

        word_counts = Counter()
        n_chars = 0
        for text in texts:
            n_chars += len(text)
            word_counts.update(pattern.findall(text))
        if verbose:
            print(f"[BPE] {n_chars:,} characters, {len(word_counts):,} unique words "
                  f"({time.time() - t0:.0f}s)")

        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        del word_counts

        pair_counts = defaultdict(int)
        pair_where = defaultdict(set)
        for wi, (w, f) in enumerate(zip(words, freqs)):
            for a, b in zip(w, w[1:]):
                pair_counts[(a, b)] += f
                pair_where[(a, b)].add(wi)
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        n_merges = vocab_size - 257
        merges = []
        while len(merges) < n_merges and heap:
            neg_c, pair = heapq.heappop(heap)
            count = pair_counts.get(pair, 0)
            if count <= 0:
                continue
            if -neg_c != count:
                heapq.heappush(heap, (-count, pair))
                continue
            if count < min_frequency:
                if verbose:
                    print(f"[BPE] stopping early: best pair frequency {count} < {min_frequency}")
                break

            new_id = 256 + len(merges)
            merges.append(pair)
            a, b = pair
            changed = set()
            for wi in list(pair_where[pair]):
                w, f = words[wi], freqs[wi]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    changed.add(p)
                new_w, i = [], 0
                while i < len(w):
                    if i + 1 < len(w) and w[i] == a and w[i + 1] == b:
                        new_w.append(new_id)
                        i += 2
                    else:
                        new_w.append(w[i])
                        i += 1
                words[wi] = new_w
                for p in zip(new_w, new_w[1:]):
                    pair_counts[p] += f
                    pair_where[p].add(wi)
                    changed.add(p)
            del pair_where[pair]
            pair_counts.pop(pair, None)
            changed.discard(pair)
            for p in changed:
                c = pair_counts.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p))
                else:
                    pair_counts.pop(p, None)
                    pair_where.pop(p, None)
            if verbose and (len(merges) % 2000 == 0):
                print(f"[BPE] {len(merges):,}/{n_merges:,} merges ({time.time() - t0:.0f}s)")

        tok = cls(merges, name=name)
        if verbose:
            print(f"[BPE] done: vocab size {len(tok):,} ({time.time() - t0:.0f}s)")
        return tok

    def _encode_word(self, word):
        ids = self._cache.get(word)
        if ids is not None:
            return ids
        ids = list(word.encode("utf-8"))
        ranks = self.ranks
        while len(ids) >= 2:
            best_rank, best_pair = None, None
            for p in zip(ids, ids[1:]):
                r = ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_pair = r, p
            if best_pair is None:
                break
            a, b = best_pair
            new_id = 256 + best_rank
            merged, i = [], 0
            while i < len(ids):
                if i + 1 < len(ids) and ids[i] == a and ids[i + 1] == b:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(ids[i])
                    i += 1
            ids = merged
        if len(self._cache) > 2_000_000:
            self._cache.clear()
        self._cache[word] = ids
        return ids

    def encode(self, text, add_special_tokens=False):
        out = []
        for word in self._pattern.findall(text):
            out.extend(self._encode_word(word))
        return out

    def __call__(self, texts, add_special_tokens=False, **kwargs):
        if isinstance(texts, str):
            return {"input_ids": self.encode(texts)}
        return {"input_ids": [self.encode(t) for t in texts]}

    def decode(self, ids, skip_special_tokens=True, **kwargs):
        parts = []
        for i in ids:
            i = int(i)
            if i == self.eos_token_id:
                if not skip_special_tokens:
                    parts.append(self.eos_token.encode("utf-8"))
                continue
            b = self.vocab.get(i)
            if b is not None:
                parts.append(b)
        return b"".join(parts).decode("utf-8", errors="replace")

    def batch_decode(self, sequences, skip_special_tokens=True, **kwargs):
        return [self.decode(s, skip_special_tokens=skip_special_tokens) for s in sequences]

    def convert_ids_to_tokens(self, ids):
        return [self.vocab[i].decode("utf-8", errors="replace") if i in self.vocab else self.eos_token
                for i in ids]

    def save_pretrained(self, directory):
        os.makedirs(directory, exist_ok=True)
        path = os.path.join(directory, "legal_bpe_tokenizer.json")
        with open(path, "w") as f:
            json.dump({
                "type": "byte-level-bpe",
                "name": self.name,
                "pattern": PRETOKENIZE_PATTERN,
                "eos_token": self.eos_token,
                "eos_token_id": self.eos_token_id,
                "vocab_size": len(self),
                "merges": [list(m) for m in self.merges],
            }, f)
        return path

    @classmethod
    def from_pretrained(cls, directory):
        path = directory if directory.endswith(".json") else os.path.join(directory, "legal_bpe_tokenizer.json")
        with open(path) as f:
            data = json.load(f)
        assert data["pattern"] == PRETOKENIZE_PATTERN, "tokenizer was saved with a different pre-tokenisation pattern"
        tok = cls(data["merges"], name=data.get("name", "legal-bpe"))
        assert tok.eos_token_id == data["eos_token_id"]
        return tok

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_positions, theta=10_000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_positions, dtype=torch.float32), inv_freq)
        self.register_buffer("cos_cached", torch.cat([freqs.cos(), freqs.cos()], dim=-1), persistent=False)
        self.register_buffer("sin_cached", torch.cat([freqs.sin(), freqs.sin()], dim=-1), persistent=False)

    def forward(self, x, offset=0):
        T = x.size(-2)
        cos = self.cos_cached[offset:offset + T].to(x.dtype)
        sin = self.sin_cached[offset:offset + T].to(x.dtype)
        x1, x2 = x.chunk(2, dim=-1)
        return x * cos + torch.cat((-x2, x1), dim=-1) * sin

class SelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.0):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout

    def forward(self, x, rope, attn_mask=None, is_causal=True, cache=None, offset=0):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k = rope(q, offset), rope(k, offset)
        if cache is not None:
            if "k" in cache:
                k = torch.cat([cache["k"], k], dim=2)
                v = torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            is_causal=is_causal and attn_mask is None,
            dropout_p=self.dropout if self.training else 0.0,
        )
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, n_embd, hidden):
        super().__init__()
        self.w_gate_up = nn.Linear(n_embd, 2 * hidden, bias=False)
        self.w_down = nn.Linear(hidden, n_embd, bias=False)

    def forward(self, x):
        gate, up = self.w_gate_up(x).chunk(2, dim=-1)
        return self.w_down(F.silu(gate) * up)

class DecoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.attn = SelfAttention(n_embd, n_head, dropout)
        self.norm2 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, rope, attn_mask=None, is_causal=True, cache=None, offset=0):
        x = x + self.drop(self.attn(self.norm1(x), rope, attn_mask, is_causal, cache, offset))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x

def init_weights(model, n_layers, std=0.02):
    for module in model.modules():
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
    for name, p in model.named_parameters():
        if name.endswith("proj.weight") or name.endswith("w_down.weight"):
            nn.init.normal_(p, mean=0.0, std=std / math.sqrt(2 * n_layers))

def lm_loss(lm_head, hidden, input_ids, loss_start=0):
    start = max(int(loss_start), 1)
    logits = lm_head(hidden[:, start - 1:-1])
    targets = input_ids[:, start:]
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

class DecoderOnlyLM(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, L, H = cfg["n_embd"], cfg["n_layer"], cfg["n_head"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.blocks = nn.ModuleList(
            [DecoderBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(L)]
        )
        self.norm = RMSNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, L)

    def _hidden(self, input_ids, caches=None, offset=0, is_causal=True):
        x = self.drop(self.embed(input_ids))
        for i, block in enumerate(self.blocks):
            x = block(x, self.rope, is_causal=is_causal,
                      cache=None if caches is None else caches[i], offset=offset)
        return self.norm(x)

    def forward(self, input_ids, prefix_len=0):
        return lm_loss(self.lm_head, self._hidden(input_ids), input_ids, prefix_len)

    @torch.no_grad()
    def prefill(self, prompt_ids):
        caches = [dict() for _ in self.blocks]
        h = self._hidden(prompt_ids, caches, offset=0, is_causal=True)
        return self.lm_head(h[:, -1]), {"caches": caches, "pos": prompt_ids.size(1)}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self._hidden(token_ids[:, None], state["caches"], offset=state["pos"], is_causal=False)
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        ids = torch.cat([prompt_ids, cont_ids], dim=1)
        h = self._hidden(ids)
        return self.lm_head(h[:, prompt_ids.size(1) - 1:-1])

def prefix_lm_mask(T, prefix_len, device):
    i = torch.arange(T, device=device)
    mask = (i[None, :] <= i[:, None]) | (i[None, :] < prefix_len)
    return mask[None, None]

class PrefixLM(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, L, H = cfg["n_embd"], cfg["n_layer"], cfg["n_head"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.segment = nn.Embedding(2, d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.blocks = nn.ModuleList(
            [DecoderBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(L)]
        )
        self.norm = RMSNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, L)

    def _hidden(self, input_ids, prefix_len, caches=None, offset=0, decoding=False):
        B, T = input_ids.shape
        positions = torch.arange(offset, offset + T, device=input_ids.device)
        segments = (positions >= prefix_len).long()
        x = self.drop(self.embed(input_ids) + self.segment(segments)[None])

        if decoding:
            attn_mask, is_causal = None, False
        elif prefix_len >= T:
            attn_mask, is_causal = None, False
        elif prefix_len <= 0:
            attn_mask, is_causal = None, True
        else:
            attn_mask, is_causal = prefix_lm_mask(T, prefix_len, input_ids.device), False

        for i, block in enumerate(self.blocks):
            x = block(x, self.rope, attn_mask=attn_mask, is_causal=is_causal,
                      cache=None if caches is None else caches[i], offset=offset)
        return self.norm(x)

    def forward(self, input_ids, prefix_len):
        P = int(prefix_len)
        return lm_loss(self.lm_head, self._hidden(input_ids, P), input_ids, P)

    @torch.no_grad()
    def prefill(self, prompt_ids):
        P = prompt_ids.size(1)
        caches = [dict() for _ in self.blocks]
        h = self._hidden(prompt_ids, prefix_len=P, caches=caches, offset=0)
        return self.lm_head(h[:, -1]), {"caches": caches, "pos": P, "prefix_len": P}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self._hidden(token_ids[:, None], state["prefix_len"], state["caches"],
                         offset=state["pos"], decoding=True)
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        P = prompt_ids.size(1)
        h = self._hidden(torch.cat([prompt_ids, cont_ids], dim=1), prefix_len=P)
        return self.lm_head(h[:, P - 1:-1])

class CrossAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.0):
        super().__init__()
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.q = nn.Linear(n_embd, n_embd, bias=False)
        self.kv = nn.Linear(n_embd, 2 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout

    def forward(self, x, memory, cache=None):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        if cache is not None and "k" in cache:
            k, v = cache["k"], cache["v"]
        else:
            S = memory.size(1)
            k, v = self.kv(memory).view(B, S, 2, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
            if cache is not None:
                cache["k"], cache["v"] = k, v
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class EncoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.attn = SelfAttention(n_embd, n_head, dropout)
        self.norm2 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, rope):
        x = x + self.drop(self.attn(self.norm1(x), rope, is_causal=False))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x

class CrossDecoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.self_attn = SelfAttention(n_embd, n_head, dropout)
        self.norm2 = RMSNorm(n_embd)
        self.cross_attn = CrossAttention(n_embd, n_head, dropout)
        self.norm3 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, memory, rope, cache=None, offset=0, is_causal=True):
        self_cache = None if cache is None else cache["self"]
        cross_cache = None if cache is None else cache["cross"]
        x = x + self.drop(self.self_attn(self.norm1(x), rope, is_causal=is_causal, cache=self_cache, offset=offset))
        x = x + self.drop(self.cross_attn(self.norm2(x), memory, cross_cache))
        x = x + self.drop(self.mlp(self.norm3(x)))
        return x

class EncoderDecoderLM(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, H = cfg["n_embd"], cfg["n_head"]
        self.bos_id = cfg["bos_token_id"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.enc_blocks = nn.ModuleList(
            [EncoderBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(cfg["n_enc_layer"])]
        )
        self.enc_norm = RMSNorm(d)
        self.dec_blocks = nn.ModuleList(
            [CrossDecoderBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(cfg["n_dec_layer"])]
        )
        self.dec_norm = RMSNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, cfg["n_enc_layer"] + cfg["n_dec_layer"])

    def encode(self, src_ids):
        x = self.drop(self.embed(src_ids))
        for block in self.enc_blocks:
            x = block(x, self.rope)
        return self.enc_norm(x)

    def decode(self, tgt_in, memory, caches=None, offset=0, is_causal=True):
        x = self.drop(self.embed(tgt_in))
        for i, block in enumerate(self.dec_blocks):
            x = block(x, memory, self.rope, None if caches is None else caches[i], offset, is_causal)
        return self.dec_norm(x)

    def _shift_right(self, tgt):
        bos = torch.full_like(tgt[:, :1], self.bos_id)
        return torch.cat([bos, tgt[:, :-1]], dim=1)

    def forward(self, input_ids, prefix_len):
        P = int(prefix_len)
        assert 0 < P < input_ids.size(1), "encoder-decoder needs 0 < prefix_len < block size"
        src, tgt = input_ids[:, :P], input_ids[:, P:]
        h = self.decode(self._shift_right(tgt), self.encode(src))
        logits = self.lm_head(h)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))

    @torch.no_grad()
    def prefill(self, prompt_ids):
        memory = self.encode(prompt_ids)
        caches = [{"self": {}, "cross": {}} for _ in self.dec_blocks]
        bos = torch.full((prompt_ids.size(0), 1), self.bos_id, dtype=torch.long, device=prompt_ids.device)
        h = self.decode(bos, memory, caches, offset=0, is_causal=True)
        return self.lm_head(h[:, -1]), {"memory": memory, "caches": caches, "pos": 1}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self.decode(token_ids[:, None], state["memory"], state["caches"], offset=state["pos"], is_causal=False)
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        return self.lm_head(self.decode(self._shift_right(cont_ids), self.encode(prompt_ids)))

class MultiScaleRetention(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.qkvg = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        gammas = 1.0 - 2.0 ** (-5.0 - torch.arange(n_head, dtype=torch.float32))
        self.register_buffer("log_gamma", torch.log(gammas), persistent=False)

    def _decay_matrix(self, T, device):
        idx = torch.arange(T, device=device, dtype=torch.float32)
        diff = idx[:, None] - idx[None, :]
        decay = torch.exp(diff.clamp(min=0)[None] * self.log_gamma[:, None, None])
        decay = decay.masked_fill(diff[None] < 0, 0.0)
        return decay / decay.sum(dim=-1, keepdim=True).sqrt()

    def forward(self, x, rope, state=None, offset=0):
        B, T, C = x.shape
        q, k, v, g = self.qkvg(x).split(C, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k = rope(q, offset), rope(k, offset)
        k = k * self.head_dim ** -0.5

        if state is not None and "S" in state:
            with torch.autocast(device_type=x.device.type, enabled=False):
                gamma = self.log_gamma.exp().view(1, -1, 1, 1)
                state["S"] = gamma * state["S"] + k.float().transpose(-1, -2) @ v.float()
                state["scale"] = gamma * state["scale"] + 1.0
                y = ((q.float() @ state["S"]) / state["scale"].sqrt()).to(v.dtype)
        else:
            decay = self._decay_matrix(T, x.device).to(q.dtype)
            y = ((q @ k.transpose(-1, -2)) * decay) @ v
            if state is not None:
                with torch.autocast(device_type=x.device.type, enabled=False):
                    pos = torch.arange(T, device=x.device, dtype=torch.float32)
                    w = torch.exp((T - 1 - pos)[None, :] * self.log_gamma[:, None])
                    state["S"] = torch.einsum("bhtd,ht,bhte->bhde", k.float(), w, v.float())
                    state["scale"] = w.sum(dim=-1).view(1, -1, 1, 1)

        y = y.transpose(1, 2).float()
        y = y * torch.rsqrt(y.pow(2).mean(dim=-1, keepdim=True) + 1e-6)
        y = y.reshape(B, T, C).to(g.dtype)
        return self.proj(F.silu(g) * y)

class FeedForward(nn.Module):
    def __init__(self, n_embd, hidden):
        super().__init__()
        self.w_in = nn.Linear(n_embd, hidden)
        self.w_down = nn.Linear(hidden, n_embd)

    def forward(self, x):
        return self.w_down(F.gelu(self.w_in(x)))

class RetNetBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_embd)
        self.retention = MultiScaleRetention(n_embd, n_head)
        self.norm2 = nn.LayerNorm(n_embd)
        self.ffn = FeedForward(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, rope, state=None, offset=0):
        x = x + self.drop(self.retention(self.norm1(x), rope, state, offset))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x

class RetNetLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, L, H = cfg["n_embd"], cfg["n_layer"], cfg["n_head"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.blocks = nn.ModuleList([RetNetBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(L)])
        self.norm = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, L)

    def _hidden(self, input_ids, states=None, offset=0):
        x = self.drop(self.embed(input_ids))
        for i, block in enumerate(self.blocks):
            x = block(x, self.rope, None if states is None else states[i], offset)
        return self.norm(x)

    def forward(self, input_ids, prefix_len=0):
        return lm_loss(self.lm_head, self._hidden(input_ids), input_ids, prefix_len)

    @torch.no_grad()
    def prefill(self, prompt_ids):
        states = [dict() for _ in self.blocks]
        h = self._hidden(prompt_ids, states, offset=0)
        return self.lm_head(h[:, -1]), {"states": states, "pos": prompt_ids.size(1)}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self._hidden(token_ids[:, None], state["states"], offset=state["pos"])
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        ids = torch.cat([prompt_ids, cont_ids], dim=1)
        return self.lm_head(self._hidden(ids)[:, prompt_ids.size(1) - 1:-1])

ARCH_CLASSES = {'decoder_llama': DecoderOnlyLM, 'hybrid_prefixlm': PrefixLM, 'encdec_t5': EncoderDecoderLM, 'efficient_retnet': RetNetLM}

### Backbone wrapper + fine-tuning methods (LoRMA, IA3, prompt, partial, ...; from Phases 2-5)

In [ ]:
import math, time, json, os
import torch
import torch.nn as nn
import torch.nn.functional as F

class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale

class RSLoRALinear(LoRALinear):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__(base, r, alpha, dropout)
        self.scale = alpha / math.sqrt(r)

class IA3Linear(nn.Module):
    def __init__(self, base, mode, n_split=3):
        super().__init__()
        self.base, self.mode = base, mode
        if mode == "out_kv":
            d = base.out_features // n_split
            self.d = d
            self.l_kv = nn.Parameter(torch.ones(2 * d))
        else:
            self.l_in = nn.Parameter(torch.ones(base.in_features))
    def forward(self, x):
        if self.mode == "in":
            return self.base(x * self.l_in)
        y = self.base(x)
        d = self.d
        scale = torch.cat([torch.ones(d, device=y.device, dtype=y.dtype), self.l_kv.to(y.dtype),
                           torch.ones(y.size(-1) - 3 * d, device=y.device, dtype=y.dtype)])
        return y * scale

class LoRMALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        y = self.base(x)
        return y + (self.drop(y) @ self.A.t() @ self.B.t()) * self.scale

class AdapterLinear(nn.Module):
    def __init__(self, base, bottleneck=64):
        super().__init__()
        self.base = base
        self.down = nn.Linear(base.out_features, bottleneck)
        self.up = nn.Linear(bottleneck, base.out_features)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        y = self.base(x)
        return y + self.up(F.gelu(self.down(y)))

class BiasLinear(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.bias = nn.Parameter(torch.zeros(base.out_features))
    def forward(self, x):
        return self.base(x) + self.bias

ATTN_LINEARS = ("qkv", "proj", "qkvg", "q", "kv")
FFN_LINEARS = ("w_gate_up", "w_down", "w_in")
OUT_PROJ = ("proj", "w_down")

def _replace_linears(module, names, make):
    n = 0
    for parent in list(module.modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and cname in names:
                setattr(parent, cname, make(child)); n += 1
    return n

def backbone_blocks(arch, bb):
    return list(bb.enc_blocks) if arch == "encdec_t5" else list(bb.blocks)

class NERModel(nn.Module):
    def __init__(self, arch, backbone, n_tags, context="bidir", head="linear", n_prompt=0, dropout=0.1):
        super().__init__()
        self.arch, self.bb, self.context = arch, backbone, context
        self.head_out = n_tags
        d = backbone.cfg["n_embd"]
        self.drop = nn.Dropout(dropout)
        if head == "mlp":
            self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(dropout), nn.Linear(d, n_tags))
        else:
            self.head = nn.Linear(d, n_tags)
        self.prompt = nn.Parameter(torch.randn(n_prompt, d) * 0.02) if n_prompt > 0 else None
        if hasattr(backbone, "lm_head"):
            backbone.lm_head = nn.Identity()
        if arch == "encdec_t5":
            backbone.dec_blocks = nn.ModuleList(); backbone.dec_norm = nn.Identity()

    def _mask(self, key_ok, T, causal):
        m = key_ok[:, None, None, :].expand(-1, 1, T, T)
        if causal:
            m = m & torch.ones(T, T, dtype=torch.bool, device=key_ok.device).tril()[None, None]
        eye = torch.eye(T, dtype=torch.bool, device=key_ok.device)[None, None]
        return m | eye

    def hidden(self, ids, attn):
        bb, arch = self.bb, self.arch
        B, T = ids.shape
        x = bb.embed(ids)
        ok = attn.bool()
        if self.context == "echo" and arch in ("decoder_llama", "efficient_retnet"):
            x = torch.cat([x, x], dim=1); ok = torch.cat([ok, ok], dim=1)
        if arch == "hybrid_prefixlm":
            x = x + bb.segment(torch.zeros(x.size(1), dtype=torch.long, device=x.device))[None]
        P = 0
        if self.prompt is not None:
            P = self.prompt.size(0)
            x = torch.cat([self.prompt[None].expand(B, -1, -1).to(x.dtype), x], dim=1)
            ok = torch.cat([torch.ones(B, P, dtype=torch.bool, device=x.device), ok], dim=1)
        x = bb.drop(x)
        L = x.size(1)
        if arch == "efficient_retnet":
            for blk in bb.blocks:
                x = blk(x, bb.rope)
            x = bb.norm(x)
        elif arch == "encdec_t5":
            mask = self._mask(ok, L, causal=False)
            for blk in bb.enc_blocks:
                x = x + blk.drop(blk.attn(blk.norm1(x), bb.rope, attn_mask=mask, is_causal=False))
                x = x + blk.drop(blk.mlp(blk.norm2(x)))
            x = bb.enc_norm(x)
        else:
            causal = self.context in ("causal", "echo") and arch == "decoder_llama"
            mask = self._mask(ok, L, causal=causal)
            for blk in bb.blocks:
                x = blk(x, bb.rope, attn_mask=mask, is_causal=False)
            x = bb.norm(x)
        x = x[:, P:]
        if self.context == "echo" and arch in ("decoder_llama", "efficient_retnet"):
            x = x[:, T:]
        return x

    def forward(self, ids, attn):
        return self.head(self.drop(self.hidden(ids, attn)))

def load_backbone(model_dir, arch_classes, map_location="cpu"):
    cfg = json.load(open(os.path.join(model_dir, "config.json")))
    bb = arch_classes[cfg["arch"]](cfg)
    sd = torch.load(os.path.join(model_dir, "pytorch_model.bin"), map_location=map_location)
    missing, unexpected = bb.load_state_dict(sd, strict=False)
    missing = [k for k in missing if not k.endswith("lm_head.weight")]
    assert not missing and not unexpected, (missing, unexpected)
    return cfg, bb

def keep_layers(arch, bb, keep):
    if arch == "encdec_t5":
        bb.enc_blocks = nn.ModuleList([bb.enc_blocks[i] for i in keep]); bb.cfg["n_enc_layer"] = len(keep)
    else:
        bb.blocks = nn.ModuleList([bb.blocks[i] for i in keep]); bb.cfg["n_layer"] = len(keep)
    return bb

def apply_method(model, method, C):
    arch, bb = model.arch, model.bb
    info = {}
    if method in ("full", "kd"):
        for p in model.parameters(): p.requires_grad = True
    else:
        for p in bb.parameters(): p.requires_grad = False
        if method == "partial":
            blocks = backbone_blocks(arch, bb)
            for blk in blocks[-C["PARTIAL_TOP_K"]:]:
                for p in blk.parameters(): p.requires_grad = True
            norm = bb.enc_norm if arch == "encdec_t5" else bb.norm
            for p in norm.parameters(): p.requires_grad = True
        elif method in ("lora", "lorma", "rslora"):
            cls = {"lora": LoRALinear, "lorma": LoRMALinear, "rslora": RSLoRALinear}[method]
            targets = ATTN_LINEARS + (FFN_LINEARS if C["PEFT_TARGET_FFN"] else ())
            info["wrapped"] = _replace_linears(bb, targets, lambda l: cls(l, C["RANK"], C["ALPHA"], C["PEFT_DROPOUT"]))
        elif method == "adapter":
            info["wrapped"] = _replace_linears(bb, OUT_PROJ, lambda l: AdapterLinear(l, C["ADAPTER_DIM"]))
        elif method == "bitfit":
            info["wrapped"] = _replace_linears(bb, ATTN_LINEARS + FFN_LINEARS, BiasLinear)
            for n, p in bb.named_parameters():
                if n.endswith("bias") or "norm" in n: p.requires_grad = True
        elif method == "gradual":
            pass
        elif method == "ia3":
            n = _replace_linears(bb, ("qkv",), lambda l: IA3Linear(l, "out_kv", 3))
            n += _replace_linears(bb, ("qkvg",), lambda l: IA3Linear(l, "out_kv", 4))
            n += _replace_linears(bb, ("w_down",), lambda l: IA3Linear(l, "in"))
            info["wrapped"] = n
        elif method == "prompt":
            pass
        else:
            raise ValueError(method)
        for p in model.head.parameters(): p.requires_grad = True
        if model.prompt is not None: model.prompt.requires_grad = True
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    info.update(trainable_params=n_train, total_params=n_total, trainable_pct=100.0 * n_train / n_total)
    return info

### Retrieval: data, BM25, dual encoder, contrastive training, KD, fusion, metrics

In [ ]:
import json, os, re, math, time, random
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def _text(x):
    t = x.get("text", x.get("document", x.get("sentences", ""))) if isinstance(x, dict) else x
    return " ".join(map(str, t)) if isinstance(t, list) else str(t)

def parse_cases(obj, id_keys=("id", "query_id", "candidate_id", "case_id", "doc_id")):
    items = obj.items() if isinstance(obj, dict) else [(None, x) for x in obj]
    out = []
    for k, d in items:
        cid = k
        if isinstance(d, dict):
            for key in id_keys:
                if key in d: cid = d[key]; break
        rel = d.get("relevant_candidates", d.get("relevant", d.get("labels"))) if isinstance(d, dict) else None
        out.append({"id": str(cid), "text": _text(d), "relevant": [str(r) for r in rel] if rel is not None else None})
    return out

_WORD = re.compile(r"[a-z][a-z0-9]+|\d+")
STOP = set("the of and to in a is that for be by on as was with it this which are or at from an not has have had been were "
           "his her he she they their its any shall may such also there said would no into these those under upon being who whom "
           "than other so if all them can will only one two citation".split())
def lex_tokens(text):
    return [w for w in _WORD.findall(text.lower()) if w not in STOP]

class BM25:
    def __init__(self, cand_texts, k1=1.5, b=0.75, ngram=1):
        from sklearn.feature_extraction.text import CountVectorizer
        self.vec = CountVectorizer(analyzer=lambda t: self._grams(t, ngram), min_df=2 if len(cand_texts) > 50 else 1)
        X = self.vec.fit_transform(cand_texts).tocsr().astype(np.float32)
        n = X.shape[0]; df = np.bincount(X.indices, minlength=X.shape[1])
        self.idf = np.log(1 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)
        dl = np.asarray(X.sum(1)).ravel(); avg = dl.mean()
        rows = np.repeat(np.arange(n), np.diff(X.indptr))
        tf = X.data; X.data = tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[rows] / avg)) * self.idf[X.indices]
        self.W = X
    @staticmethod
    def _grams(text, ngram):
        t = lex_tokens(text)
        return t + ([a + "_" + c for a, c in zip(t, t[1:])] if ngram >= 2 else [])
    def scores(self, query_texts, qweight="log"):
        Q = self.vec.transform(query_texts).tocsr().astype(np.float32)
        Q.data = np.log1p(Q.data) if qweight == "log" else np.ones_like(Q.data)
        return (Q @ self.W.T).toarray()

def rank_top(scores, cand_ids, query_ids, k=7):
    out = {}
    for i, q in enumerate(query_ids):
        s = scores[i].astype(np.float64).copy()
        for j, c in enumerate(cand_ids):
            if c == q: s[j] = -np.inf
        top = np.argsort(-s, kind="stable")[:k]
        out[q] = [cand_ids[j] for j in top]
    return out

def f1_at_k(pred, gold, k=7):
    tp = sum(len(set(pred[q][:k]) & set(gold[q])) for q in gold)
    n_pred = sum(min(k, len(pred[q])) for q in gold); n_gold = sum(len(gold[q]) for q in gold)
    P = tp / max(1, n_pred); R = tp / max(1, n_gold)
    return {"micro_F1@7": 2 * P * R / (P + R) if P + R else 0.0, "micro_P@7": P, "micro_R@7": R}

def zscore_rows(s):
    return (s - s.mean(1, keepdims=True)) / (s.std(1, keepdims=True) + 1e-9)

def fuse(dense, lex, w):
    return w * zscore_rows(dense) + (1 - w) * zscore_rows(lex)

def doc_windows(text, C, n):
    wc = int(C["CHUNK_LEN"] * C["CHARS_PER_TOKEN"])
    wins = [text[i:i + wc] for i in range(0, max(len(text), 1), wc)] or [""]
    if len(wins) <= n: return wins
    idx = sorted({round(i * (len(wins) - 1) / (n - 1)) for i in range(n)}) if n > 1 else [0]
    return [wins[i] for i in idx]

def encode_docs(docs, tok, C):
    out = {}
    for i, t in docs.items():
        ch = [tok.encode(w)[: C["CHUNK_LEN"]] for w in doc_windows(t, C, C["MAX_CHUNKS"])]
        out[i] = [c for c in ch if c] or [[tok.eos_token_id]]
    return out

def pack(chunk_lists, pad_id, device):
    flat = [c for cs in chunk_lists for c in cs]; owner = [d for d, cs in enumerate(chunk_lists) for _ in cs]
    L = max(len(c) for c in flat)
    ids = torch.full((len(flat), L), pad_id, dtype=torch.long); att = torch.zeros((len(flat), L), dtype=torch.long)
    for r, c in enumerate(flat): ids[r, :len(c)] = torch.tensor(c); att[r, :len(c)] = 1
    return ids.to(device), att.to(device), torch.tensor(owner, device=device)

class RetrModel(NERModel):
    def __init__(self, arch, backbone, context, pool="mean", proj_dim=256, n_prompt=0):
        super().__init__(arch, backbone, 1, context=context, head="linear", n_prompt=n_prompt, dropout=0.0)
        d = backbone.cfg["n_embd"]
        self.pool = pool
        self.attn = nn.Linear(d, 1)
        self.proj = nn.Linear(d, proj_dim)
        self.head = nn.ModuleList([self.attn, self.proj])

    def encode(self, ids, att, owner, n_docs):
        h = self.hidden(ids, att)
        a = att.unsqueeze(-1).to(h.dtype)
        cv = ((h * a).sum(1) / a.sum(1).clamp(min=1)).float()
        with torch.autocast(device_type=cv.device.type, enabled=False):
            if self.pool == "attn":
                s = self.attn(cv).squeeze(-1); s = s - s.max()
                w = torch.exp(s); den = torch.zeros(n_docs, device=cv.device).index_add_(0, owner, w)
                doc = torch.zeros(n_docs, cv.size(1), device=cv.device).index_add_(0, owner, cv * w[:, None]) / den[:, None]
            else:
                cnt = torch.zeros(n_docs, device=cv.device).index_add_(0, owner, torch.ones_like(owner, dtype=cv.dtype))
                doc = torch.zeros(n_docs, cv.size(1), device=cv.device).index_add_(0, owner, cv) / cnt[:, None]
            return F.normalize(self.proj(doc), dim=-1)

@torch.no_grad()
def embed_all(model, chunks, C, pad_id, device, log=None):
    model.eval(); keys = list(chunks); vecs = []; bs = C["EMBED_BATCH_DOCS"]
    for i in range(0, len(keys), bs):
        part = [chunks[k] for k in keys[i:i + bs]]
        ids, att, owner = pack(part, pad_id, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=C["AMP"] and device.type == "cuda"):
            vecs.append(model.encode(ids, att, owner, len(part)).float().cpu())
    return keys, torch.cat(vecs).numpy()

def contrastive_loss(qv, cv, labels, mask, tau):
    logits = (qv @ cv.T) / tau
    logits = logits.masked_fill(mask, -1e4)
    return F.cross_entropy(logits, labels), logits

def kd_loss(s_logits, t_logits, T):
    return F.kl_div(F.log_softmax(s_logits / T, -1), F.softmax(t_logits / T, -1), reduction="batchmean") * T * T

def make_train_items(queries, rel, hard_negs, rng, C):
    items = []
    for q in queries:
        for p in rel[q]:
            hn = hard_negs.get(q) or []
            items.append((q, p, rng.choice(hn[: C["HARD_NEG_TOP"]]) if hn else None))
    rng.shuffle(items)
    return items

def train(model, q_chunks, c_chunks, rel, hard_negs, eval_fn, C, pad_id, device, teacher=None, log=print):
    gradual = C["METHOD"] == "gradual"
    params = list(model.parameters()) if gradual else [p for p in model.parameters() if p.requires_grad]
    head_ids = {id(p) for p in model.head.parameters()}
    if model.prompt is not None: head_ids.add(id(model.prompt))
    groups = [g for g in ({"params": [p for p in params if id(p) in head_ids], "lr": C["HEAD_LR"]},
                          {"params": [p for p in params if id(p) not in head_ids], "lr": C["LR"]}) if g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=C["WEIGHT_DECAY"])
    rng = random.Random(C["SEED"]); qs = [q for q in rel if rel[q] and q in q_chunks]
    n_items = sum(len(rel[q]) for q in qs); steps_per_epoch = math.ceil(n_items / C["BATCH_QUERIES"])
    total = steps_per_epoch * C["EPOCHS"]; warm = int(C["WARMUP_FRAC"] * total); base = [g["lr"] for g in opt.param_groups]
    amp = C["AMP"] and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    best, best_state, hist, step, t0 = -1.0, None, [], 0, time.time()
    n_blocks = len(backbone_blocks(model.arch, model.bb)); time_up = False
    def sample(chs):
        return chs if len(chs) <= C["TRAIN_CHUNKS"] else [chs[i] for i in sorted(rng.sample(range(len(chs)), C["TRAIN_CHUNKS"]))]
    for ep in range(1, C["EPOCHS"] + 1):
        if time_up: break
        if gradual:
            k = max(1, round(n_blocks * ep / C["EPOCHS"])) if ep < C["EPOCHS"] else n_blocks
            unfreeze_top(model, k); log(f"  gradual unfreezing: top {k}/{n_blocks} layers trainable")
        model.train(); items = make_train_items(qs, rel, hard_negs, rng, C); run = 0.0
        for bi in range(0, len(items), C["BATCH_QUERIES"]):
            batch = items[bi:bi + C["BATCH_QUERIES"]]
            qids = [x[0] for x in batch]
            cands = [x[1] for x in batch] + [x[2] for x in batch if x[2] is not None]
            q_ids, q_att, q_own = pack([sample(q_chunks[q]) for q in qids], pad_id, device)
            c_ids, c_att, c_own = pack([sample(c_chunks[c]) for c in cands], pad_id, device)
            labels = torch.arange(len(batch), device=device)
            mask = torch.tensor([[j != i and cands[j] in rel[q] for j in range(len(cands))] for i, q in enumerate(qids)], device=device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                qv = model.encode(q_ids, q_att, q_own, len(qids)); cv = model.encode(c_ids, c_att, c_own, len(cands))
                loss, lg = contrastive_loss(qv.float(), cv.float(), labels, mask, C["TAU"])
                if teacher is not None:
                    with torch.no_grad():
                        tq = teacher.encode(q_ids, q_att, q_own, len(qids)); tc = teacher.encode(c_ids, c_att, c_own, len(cands))
                        tl = ((tq.float() @ tc.float().T) / C["TAU"]).masked_fill(mask, -1e4)
                    loss = (1 - C["KD_ALPHA"]) * loss + C["KD_ALPHA"] * kd_loss(lg, tl, C["KD_TEMPERATURE"])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            for g, b in zip(opt.param_groups, base):
                g["lr"] = b * (step / max(1, warm) if step < warm else max(0.0, (total - step) / max(1, total - warm)))
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_([p for p in params if p.grad is not None], C["GRAD_CLIP"])
            scaler.step(opt); scaler.update(); step += 1; run += loss.item()
            if step % C["LOG_EVERY"] == 0:
                log(f"  epoch {ep} step {step}/{total} loss {run / C['LOG_EVERY']:.4f} ({time.time() - t0:.0f}s)"); run = 0.0
            if (time.time() - t0) / 60 > C["MAX_TRAIN_MINUTES"]:
                log(f"  time cap of {C['MAX_TRAIN_MINUTES']} min reached at step {step}/{total}"); time_up = True; break
        res = eval_fn(model)
        hist.append({"epoch": ep, **{k: v for k, v in res.items() if isinstance(v, (int, float))}, "time_s": time.time() - t0})
        log(f"epoch {ep}: dev F1@7 dense {res['dense_F1@7']:.4f} | fused {res['fused_F1@7']:.4f} (w={res['w']:.2f}) "
            f"| BM25 {res['bm25_F1@7']:.4f} ({time.time() - t0:.0f}s)")
        if res["fused_F1@7"] > best:
            best = res["fused_F1@7"]; trainable = {n for n, p in model.named_parameters() if p.requires_grad} if not gradual else None
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if trainable is None or k in trainable}
    if best_state is not None: model.load_state_dict(best_state, strict=False)
    return {"best_dev_fused_F1@7": best, "history": hist, "train_time_s": time.time() - t0, "steps": step}

def unfreeze_top(model, n_layers):
    blocks = backbone_blocks(model.arch, model.bb)
    for i, blk in enumerate(blocks):
        for p in blk.parameters(): p.requires_grad = i >= len(blocks) - n_layers
    norm = model.bb.enc_norm if model.arch == "encdec_t5" else model.bb.norm
    for p in norm.parameters(): p.requires_grad = True

### 1. Data

In [ ]:
if SMOKE_TEST:
    _rs = random.Random(0)
    _topics = {t: " ".join(f"{t}word{k}" for k in range(12)) for t in ["land", "tax", "murder", "contract", "service", "bail"]}
    def _case(t, i): return " ".join([_topics[t]] * 3 + [f"filler sentence {i} about procedure and hearing."] * 30)
    def _split(nq, nc, off):
        cands = {f"{off + j:010d}": (_rs.choice(list(_topics))) for j in range(nc)}
        qs = []
        for i in range(nq):
            t = _rs.choice(list(_topics)); rel = [c for c, ct in cands.items() if ct == t][:5]
            qs.append({"id": f"{off + 900 + i:010d}", "text": [_case(t, i)], "relevant_candidates": rel})
        return qs, {c: {"text": [_case(t, j)]} for j, (c, t) in enumerate(cands.items())}
    _raw = {s: _split(nq, nc, off) for s, nq, nc, off in (("train", 24, 40, 0), ("dev", 8, 30, 1000), ("test", 6, 30, 2000))}
    for _q in _raw["test"][0]: _q.pop("relevant_candidates")
else:
    _raw = {s: (load_json(data_file(f"{d}/{s}.json")), load_json(data_file(f"{d}/{s}_candidates.json")))
            for s, d in (("train", "Train"), ("dev", "Dev"), ("test", "Test"))}
SPLITS = {}
for s, (qo, co) in _raw.items():
    qs, cs = parse_cases(qo), parse_cases(co)
    SPLITS[s] = {"queries": {q["id"]: q["text"] for q in qs}, "cands": {c["id"]: c["text"] for c in cs},
                 "rel": {q["id"]: q["relevant"] for q in qs if q["relevant"] is not None}}
for s, d in SPLITS.items():
    ql = sorted(len(t.split()) for t in d["queries"].values()); cl = sorted(len(t.split()) for t in d["cands"].values())
    r = [len(v) for v in d["rel"].values()]
    inpool = np.mean([all(c in d["cands"] for c in v) for v in d["rel"].values()]) if r else float("nan")
    print(f"{s}: {len(d['queries'])} queries, {len(d['cands'])} candidates | query words median {ql[len(ql)//2]:,}, candidate words "
          f"median {cl[len(cl)//2]:,} | relevant per query " + (f"mean {np.mean(r):.2f}, median {np.median(r):.0f}, max {max(r)}"
          f"; all relevant ids in pool: {inpool:.0%}; queries that are also candidates: "
          f"{np.mean([q in d['cands'] for q in d['queries']]):.0%}" if r else "(hidden)"))

t0 = time.time(); dev = SPLITS["dev"]; dcand = list(dev["cands"]); dq = list(dev["queries"])
best_bm25, grid_res = None, []
for ng in BM25_GRID["ngram"]:
    for k1 in BM25_GRID["k1"]:
        for b in BM25_GRID["b"]:
            bm = BM25([dev["cands"][c] for c in dcand], k1=k1, b=b, ngram=ng)
            for qw in BM25_GRID["qweight"]:
                f = f1_at_k(rank_top(bm.scores([dev["queries"][q] for q in dq], qw), dcand, dq), dev["rel"])["micro_F1@7"]
                grid_res.append({"ngram": ng, "k1": k1, "b": b, "qweight": qw, "F1@7": f})
                if best_bm25 is None or f > best_bm25["F1@7"]: best_bm25 = grid_res[-1]
print(f"BM25 grid on dev ({time.time() - t0:.0f}s): best {best_bm25} | worst F1@7 {min(g['F1@7'] for g in grid_res):.4f}")
for s, d in SPLITS.items():
    d["cand_ids"], d["query_ids"] = list(d["cands"]), list(d["queries"])
    bm = BM25([d["cands"][c] for c in d["cand_ids"]], best_bm25["k1"], best_bm25["b"], best_bm25["ngram"])
    d["bm25"] = bm.scores([d["queries"][q] for q in d["query_ids"]], best_bm25["qweight"])
BM25_DEV = f1_at_k(rank_top(SPLITS["dev"]["bm25"], SPLITS["dev"]["cand_ids"], SPLITS["dev"]["query_ids"]), SPLITS["dev"]["rel"])
print("BM25 dev:", {k: round(v, 4) for k, v in BM25_DEV.items()})

tr = SPLITS["train"]; HARD_NEGS = {}
for i, q in enumerate(tr["query_ids"]):
    order = np.argsort(-tr["bm25"][i])
    HARD_NEGS[q] = [tr["cand_ids"][j] for j in order[:60] if tr["cand_ids"][j] not in set(tr["rel"].get(q, [])) and tr["cand_ids"][j] != q][:HARD_NEG_TOP]
_tr_f1 = f1_at_k(rank_top(tr["bm25"], tr["cand_ids"], tr["query_ids"]), tr["rel"])["micro_F1@7"]
print(f"BM25 train F1@7 {_tr_f1:.4f} (hard negatives = its top non-relevant candidates)")

os.makedirs(os.path.join(WORK, "bm25_baseline"), exist_ok=True)
_bm_pred = rank_top(SPLITS["test"]["bm25"], SPLITS["test"]["cand_ids"], SPLITS["test"]["query_ids"], TOP_K)
json.dump(_bm_pred, open(os.path.join(WORK, "bm25_baseline", "prediction.json"), "w"))
with zipfile.ZipFile(os.path.join(WORK, "bm25_baseline", "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
    z.write(os.path.join(WORK, "bm25_baseline", "prediction.json"), "prediction.json")

if SMOKE_TEST:
    tokenizer = LegalBPETokenizer.train([_case(t, 0) for t in _topics] * 3, vocab_size=400, verbose=False)
else:
    MODEL_PATH = find_model_dir(C["ARCH"]); print("tokenizer from", MODEL_PATH)
    tokenizer = LegalBPETokenizer.from_pretrained(MODEL_PATH)
t0 = time.time()
for s, d in SPLITS.items():
    d["q_chunks"] = encode_docs(d["queries"], tokenizer, C); d["c_chunks"] = encode_docs(d["cands"], tokenizer, C)
print(f"tokenised all cases in {time.time() - t0:.0f}s | chunks per candidate (train): "
      f"{np.mean([len(v) for v in SPLITS['train']['c_chunks'].values()]):.2f}")

### 2. Experiment runner

In [ ]:
import gc, traceback

def build_retr(c):
    arch, method = c["ARCH"], c["METHOD"]
    if SMOKE_TEST:
        cfg = dict(arch=arch, vocab_size=len(tokenizer), n_embd=32, n_layer=4, n_enc_layer=4, n_dec_layer=2, n_head=4,
                   ffn_hidden=64, rope_theta=1e4, dropout=0.0, max_positions=1344,
                   bos_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id)
        bb = ARCH_CLASSES[arch](cfg)
    else:
        cfg, bb = load_backbone(find_model_dir(arch), ARCH_CLASSES)
    if method == "kd":
        bb = keep_layers(arch, bb, c["KD_KEEP_LAYERS"] if not SMOKE_TEST else [0, 2])
    m = RetrModel(arch, bb, context=c["CONTEXT"], pool=c["POOL"], proj_dim=c["PROJ_DIM"],
                  n_prompt=c["N_PROMPT"] if method == "prompt" else 0)
    info = apply_method(m, method, c)
    return m.to(device), info

def load_teacher(c):
    tdir = os.path.join(WORK, c["KD_TEACHER_RUN"]) if c.get("KD_TEACHER_RUN") else c["KD_TEACHER_DIR"]
    if not (tdir and os.path.exists(os.path.join(tdir, "retr_model.pt"))) and c.get("KD_TEACHER_DIR"):
        tdir = c["KD_TEACHER_DIR"]
    assert tdir and os.path.exists(os.path.join(tdir, "retr_model.pt")), f"teacher weights not found in {tdir}"
    tc = json.load(open(os.path.join(tdir, "run_config.json")))
    t, tinfo = build_retr(tc); t.load_state_dict(torch.load(os.path.join(tdir, "retr_model.pt"), map_location="cpu")); t.eval()
    for p in t.parameters(): p.requires_grad = False
    print(f"teacher: {tc['RUN_NAME']} ({tinfo['total_params'] / 1e6:.1f}M params)")
    return t, tc["RUN_NAME"]

def split_scores(model, split, c, pad):
    d = SPLITS[split]
    qk, qv = embed_all(model, d["q_chunks"], c, pad, device); ck, cv = embed_all(model, d["c_chunks"], c, pad, device)
    qi = {k: i for i, k in enumerate(qk)}; ci = {k: i for i, k in enumerate(ck)}
    Q = qv[[qi[q] for q in d["query_ids"]]]; Cm = cv[[ci[x] for x in d["cand_ids"]]]
    return Q @ Cm.T

def dev_eval(model, c, pad):
    d = SPLITS["dev"]; dense = split_scores(model, "dev", c, pad)
    rk = lambda s: rank_top(s, d["cand_ids"], d["query_ids"], c["TOP_K"])
    res = {"dense": f1_at_k(rk(dense), d["rel"]), "bm25": BM25_DEV}
    by_w = {w: f1_at_k(rk(fuse(dense, d["bm25"], w)), d["rel"]) for w in c["FUSION_W_GRID"]}
    w = max(by_w, key=lambda x: by_w[x]["micro_F1@7"])
    return {"dense_F1@7": res["dense"]["micro_F1@7"], "fused_F1@7": by_w[w]["micro_F1@7"], "bm25_F1@7": BM25_DEV["micro_F1@7"],
            "w": w, "fused": by_w[w], "dense": res["dense"], "by_w": {str(k): v["micro_F1@7"] for k, v in by_w.items()},
            "dense_scores": dense}

def run_experiment(c):
    random.seed(c["SEED"]); np.random.seed(c["SEED"]); torch.manual_seed(c["SEED"])
    out = os.path.join(WORK, c["RUN_NAME"]); os.makedirs(out, exist_ok=True)
    pad = tokenizer.eos_token_id
    model, info = build_retr(c)
    print(f"{c['ARCH']} + {c['METHOD']} (pool {c['POOL']}, lr {c['LR']}): trainable {info['trainable_params'] / 1e6:.2f}M / "
          f"{info['total_params'] / 1e6:.2f}M ({info['trainable_pct']:.2f}%)")
    teacher, tname = load_teacher(c) if c["METHOD"] == "kd" else (None, None)
    r0 = dev_eval(model, c, pad)
    print(f"before fine-tuning (Phase-1 representations as they are): dev F1@7 dense {r0['dense_F1@7']:.4f} | "
          f"fused {r0['fused_F1@7']:.4f} (w={r0['w']})")
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    tr = SPLITS["train"]
    result = train(model, {**tr["q_chunks"], **tr["c_chunks"]}, tr["c_chunks"], tr["rel"], HARD_NEGS,
                   lambda m: dev_eval(m, c, pad), c, pad, device, teacher=teacher)
    peak = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else 0.0
    del teacher; gc.collect()
    t0 = time.time(); r = dev_eval(model, c, pad); dev_s = time.time() - t0
    print(f"\nDEV {c['RUN_NAME']}: F1@7 dense {r['dense_F1@7']:.4f} | BM25 {r['bm25_F1@7']:.4f} | fused {r['fused_F1@7']:.4f} "
          f"(w={r['w']}; P {r['fused']['micro_P@7']:.4f}, R {r['fused']['micro_R@7']:.4f}) | by w: {r['by_w']}")
    t0 = time.time(); dense_test = split_scores(model, "test", c, pad); test_s = time.time() - t0
    d = SPLITS["test"]
    pred = rank_top(fuse(dense_test, d["bm25"], r["w"]), d["cand_ids"], d["query_ids"], c["TOP_K"])
    pool = set(d["cand_ids"])
    assert len(pred) == len(d["query_ids"]) and all(len(v) == 7 and len(set(v)) == 7 and set(v) <= pool and q not in v for q, v in pred.items())
    json.dump(pred, open(os.path.join(out, "prediction.json"), "w"))
    with zipfile.ZipFile(os.path.join(out, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(out, "prediction.json"), "prediction.json")
    np.save(os.path.join(out, "dev_dense_scores.npy"), r["dense_scores"].astype(np.float16))
    np.save(os.path.join(out, "test_dense_scores.npy"), dense_test.astype(np.float16))
    json.dump({k: v for k, v in c.items() if isinstance(v, (int, float, str, bool, list, dict, type(None)))},
              open(os.path.join(out, "run_config.json"), "w"), indent=2)
    metrics = {"run": c["RUN_NAME"], "arch": c["ARCH"], "method": c["METHOD"], "pool": c["POOL"], "lr": c["LR"], "teacher": tname,
               "dev_F1@7_before_ft_dense": r0["dense_F1@7"], "dev_F1@7_dense": r["dense_F1@7"], "dev_F1@7_bm25": r["bm25_F1@7"],
               "dev_F1@7_fused": r["fused_F1@7"], "fusion_w": r["w"], "dev_fused": r["fused"], "dev_dense": r["dense"],
               "dev_F1@7_by_w": r["by_w"], "bm25_params": best_bm25, **info, "train_time_s": result["train_time_s"],
               "epochs_run": len(result["history"]), "history": result["history"], "peak_gpu_GB": peak,
               "dev_inference_s": dev_s, "test_inference_s": test_s}
    json.dump(metrics, open(os.path.join(out, "metrics.json"), "w"), indent=2)
    if c["METHOD"] == "full" or c["SAVE_WEIGHTS"] or any(e.get("KD_TEACHER_RUN") == c["RUN_NAME"] for e in EXPERIMENTS):
        torch.save(model.state_dict(), os.path.join(out, "retr_model.pt"))
    print(f"saved {out} | train {result['train_time_s'] / 60:.1f} min | peak GPU {peak:.2f} GB")
    del model; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return metrics

### 3. Run the experiments of this set

In [ ]:
run_start, summary = time.time(), []
offset = 0 if RUN_SET == "A" else len(SETS["A"])
for i, e in enumerate(EXPERIMENTS, 1):
    c = experiment_config(e)
    used = (time.time() - run_start) / 3600
    longest = max([m["train_time_s"] for m in summary if "train_time_s" in m] or [0]) / 3600
    if used + 1.3 * longest > TIME_BUDGET_HOURS:
        print(f"SKIP {c['RUN_NAME']}: time budget"); summary.append({"run": c["RUN_NAME"], "status": "skipped"}); continue
    print("\n" + "=" * 100 + f"\nEXPERIMENT {i}/{len(EXPERIMENTS)} (set {RUN_SET}): {c['RUN_NAME']}   ({used:.2f} h elapsed)\n" + "=" * 100)
    try:
        summary.append(dict(run_experiment(c), status="ok"))
    except Exception as ex:
        traceback.print_exc(); summary.append({"run": c["RUN_NAME"], "status": f"FAILED: {type(ex).__name__}: {ex}"})
        gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
print("\n" + "=" * 100 + f"\nSUMMARY (dev F1@7; BM25 alone = {BM25_DEV['micro_F1@7']:.4f})")
print(f"{'run':32s} {'before':>7s} {'dense':>7s} {'fused':>7s} {'w':>4s} {'P@7':>6s} {'R@7':>6s} {'trainable':>10s} {'total':>7s} {'min':>5s} {'GB':>5s}")
for m in summary:
    if m.get("status") != "ok": print(f"{m['run']:32s} {m['status']}"); continue
    print(f"{m['run']:32s} {m['dev_F1@7_before_ft_dense']:7.4f} {m['dev_F1@7_dense']:7.4f} {m['dev_F1@7_fused']:7.4f} {m['fusion_w']:4.1f} "
          f"{m['dev_fused']['micro_P@7']:6.4f} {m['dev_fused']['micro_R@7']:6.4f} {m['trainable_params'] / 1e6:9.2f}M "
          f"{m['total_params'] / 1e6:6.1f}M {m['train_time_s'] / 60:5.1f} {m['peak_gpu_GB']:5.2f}")
json.dump({"bm25_dev": BM25_DEV, "bm25_params": best_bm25, "bm25_grid": grid_res, "runs": summary},
          open(os.path.join(WORK, f"phase7_summary_{RUN_SET}.json"), "w"), indent=2)
with zipfile.ZipFile(os.path.join(WORK, f"phase7_set{RUN_SET}.zip"), "w", zipfile.ZIP_DEFLATED) as z:
    z.write(os.path.join(WORK, f"phase7_summary_{RUN_SET}.json"), f"phase7_summary_{RUN_SET}.json")
    z.write(os.path.join(WORK, "bm25_baseline", "submission.zip"), "codabench/0_bm25_baseline_reference.zip")
    for i, m in enumerate(summary, 1):
        d = os.path.join(WORK, m["run"])
        if os.path.exists(os.path.join(d, "submission.zip")):
            z.write(os.path.join(d, "submission.zip"), f"codabench/{offset + i}_{m['run']}.zip")
        for f in ("metrics.json", "run_config.json", "dev_dense_scores.npy", "test_dense_scores.npy"):
            if os.path.exists(os.path.join(d, f)): z.write(os.path.join(d, f), f"{m['run']}/{f}")
print(f"Download phase7_set{RUN_SET}.zip; upload the files in its codabench/ folder to Codabench.")